# Computer Exercise 15.27 — Problem 2

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.) — 확장 사례연구
> **단원**: §15.27 Sequential Decision Making — *Cramér vs KL under a Neural (Non-Tabular) Head*
> **풀이 일자**: Day 94
> **언어**: Python 3 (NumPy / Matplotlib)


## 1. 문제 (원문)

> **Problem 2.** Day 93 (§15.26 Problem 2) used a **tabular** per-(s, a) $K$-atom logit head and
> showed that Cramér projection loss dominated KL for atom counts $K \in \{10, 21, 51\}$ inside a
> Bellman TD loop. Tabular heads have $O(SAK)$ parameters and can memorize each state's target
> distribution, which may inflate Cramér's advantage. **Rerun the comparison with a small neural
> head**: a shared trunk MLP (H=16, tanh) feeds two action-specific linear heads that produce
> $K$ logits. Train 3 seeds × 800 steps × $K \in \{10, 21, 51\}$ under each loss (Cramér CDF
> distance vs KL). Report (a) tail-8 greedy return, (b) sharpness $H(\hat p)$ of the resulting
> categorical distributions, (c) whether the direction of Day 93 P2 (Cramér > KL) survives once
> the head is not tabular.

### 한국어 풀이용 정리
Day 93 P2 결과를 non-tabular head 로 재판정. Tabular 였기 때문에 Cramér 우위가 확대된 것인지,
neural head 에서도 같은 방향이 나오는지 확인. $K \in \{10, 21, 51\}$ 축 유지.


## 2. 수학적 배경

### 2.1 Neural head
공유 트렁크 $\phi = \tanh(W_1 x + b_1) \in \mathbb{R}^H$. 각 행동 $a$ 에 대해
$$
\text{logit}(s, a) = W_2^{(a)} \phi + b_2^{(a)} \in \mathbb{R}^K,
$$
softmax 로 $\hat p(s, a) \in \Delta^{K-1}$ 을 얻고 원자 $z_1 < \dots < z_K$ 로
$Q(s, a) = \sum_k z_k \hat p_k$.

### 2.2 Bellman categorical projection
target $t = r + \gamma \max_{a'} Q(s', a')$ 를 nearest-two atom 로 투영 (C51).

### 2.3 두 손실
- **KL**: $L = -\sum_k \tilde p_k \log \hat p_k$; logit gradient $= \hat p - \tilde p$.
- **Cramér**: $L = \sum_k (P_k - M_k)^2$; softmax 야코비안 통해 logit gradient 유도.

### 2.4 지표
- tail-8 greedy return.
- Sharpness $\bar H(\hat p) = -\sum_k \hat p_k \log \hat p_k$ (평균 over starting states s=0..3).


## 3. 풀이 흐름

1. Chain MDP + neural head + categorical projection.
2. `run_pair(loss, K, seed)` — 800 step Bellman + 손실.
3. $K \in \{10, 21, 51\}$ × loss ∈ {Cramér, KL} × 3 시드.
4. 표: R & sharpness.
5. Bar chart: R vs K per loss.
6. 시각화: 임의 상태에서의 $\hat p$ 프로파일.
7. 결론.


In [1]:

import os
os.environ['MPLCONFIGDIR'] = '/tmp/mplcfg'
os.makedirs('/tmp/mplcfg', exist_ok=True)
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

class ChainMDP:
    def __init__(self, N=5, p_slip=0.10, step_r=-0.02, goal_r=1.0, rng=None):
        self.N, self.p_slip, self.step_r, self.goal_r = N, p_slip, step_r, goal_r
        self.rng = rng or np.random.default_rng(0)
    def reset(self):
        self.s = 0; return self.s
    def step(self, a):
        if self.rng.random() < self.p_slip:
            a = 1 - a
        if a == 1: self.s = min(self.s + 1, self.N - 1)
        else:      self.s = max(self.s - 1, 0)
        done = (self.s == self.N - 1)
        r = self.goal_r if done else self.step_r
        return self.s, r, done

def one_hot(s, N):
    x = np.zeros(N); x[s] = 1.0; return x


class NeuralHeadC51:
    def __init__(self, seed=0, H=16, N=5, A=2, K=10, loss='cramer', eta=0.05, gamma=0.95, eps=0.20):
        self.rng = np.random.default_rng(seed)
        self.H, self.N, self.A, self.K = H, N, A, K
        self.loss = loss; self.eta = eta; self.gamma = gamma; self.eps = eps
        self.W1 = self.rng.normal(0, 0.3, size=(N, H)); self.b1 = np.zeros(H)
        self.W2 = [self.rng.normal(0, 0.3, size=(H, K)) for _ in range(A)]
        self.b2 = [np.zeros(K) for _ in range(A)]
        self.atoms = np.linspace(-1.0, 1.0, K)
    def _phi(self, s):
        x = one_hot(s, self.N); h = np.tanh(self.W1.T @ x + self.b1); return x, h
    def _sm(self, h, a):
        z = h @ self.W2[a] + self.b2[a]; z = z - z.max()
        p = np.exp(z); p /= p.sum(); return p
    def Q(self, s):
        _, h = self._phi(s)
        return np.array([float(np.sum(self.atoms * self._sm(h, a))) for a in range(self.A)])
    def act(self, s, greedy=False):
        if not greedy and self.rng.random() < self.eps: return int(self.rng.integers(self.A))
        return int(np.argmax(self.Q(s)))
    def _proj(self, target):
        t = np.clip(target, self.atoms[0], self.atoms[-1])
        idx = np.searchsorted(self.atoms, t); idx = max(1, min(self.K - 1, idx))
        lo, hi = self.atoms[idx-1], self.atoms[idx]
        w_hi = (t - lo) / (hi - lo + 1e-12)
        tgt = np.zeros(self.K); tgt[idx-1] = 1 - w_hi; tgt[idx] = w_hi
        return tgt
    def update(self, s, a, r, sp, done):
        x, h = self._phi(s); p = self._sm(h, a)
        if done: target = r
        else:
            _, hp = self._phi(sp)
            qsp = np.array([float(np.sum(self.atoms * self._sm(hp, ap))) for ap in range(self.A)])
            target = r + self.gamma * qsp.max()
        tgt = self._proj(target)
        if self.loss == 'kl':
            grad_z = p - tgt
        else:  # cramer
            P = np.cumsum(p); T = np.cumsum(tgt)
            diff = P - T
            g_p = 2.0 * np.cumsum(diff[::-1])[::-1]
            grad_z = p * (g_p - float(np.sum(p * g_p)))
        self.W2[a] -= self.eta * np.outer(h, grad_z); self.b2[a] -= self.eta * grad_z
        grad_h = self.W2[a] @ grad_z
        d = grad_h * (1 - h**2)
        self.W1 -= self.eta * np.outer(x, d); self.b1 -= self.eta * d
    def entropy_at_states(self, states=(0,1,2,3), action=0):
        Hv = 0.0; n = 0
        for s in states:
            _, h = self._phi(s); p = self._sm(h, action)
            Hv += -float(np.sum(p * np.log(p + 1e-12))); n += 1
        return Hv / n


def run_pair(loss, K, seed, T=800, n_eval=60):
    env = ChainMDP(rng=np.random.default_rng(seed + 7))
    L = NeuralHeadC51(seed=seed, K=K, loss=loss)
    s = env.reset()
    for _ in range(T):
        a = L.act(s, greedy=False)
        sp, r, done = env.step(a)
        L.update(s, a, r, sp, done)
        s = env.reset() if done else sp
    rets = []
    for ep in range(n_eval):
        env.rng = np.random.default_rng(seed + 1000 + ep)
        s = env.reset(); total = 0.0
        for _ in range(50):
            a = L.act(s, greedy=True)
            s, r, done = env.step(a); total += r
            if done: break
        rets.append(total)
    return float(np.mean(rets[-8:])), L.entropy_at_states(), L
print("Neural C51 ready.")


/tmp/mplcfg is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-944qghba because there was an issue with the default path (/tmp/mplcfg); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Neural C51 ready.


In [2]:

SEEDS = [94201, 94202, 94203]
KS = [10, 21, 51]
LOSSES = ['cramer', 'kl']
rows = []; learners_last = {}
for K in KS:
    for loss in LOSSES:
        Rs = []; Hs = []; L_last = None
        for sd in SEEDS:
            R, He, L = run_pair(loss, K, sd, T=800)
            Rs.append(R); Hs.append(He); L_last = L
        rows.append({'K': K, 'loss': loss,
                     'R_mean': float(np.mean(Rs)), 'R_std': float(np.std(Rs)),
                     'H_mean': float(np.mean(Hs))})
        learners_last[(K, loss)] = L_last
df2 = pd.DataFrame(rows); df2


,K,loss,R_mean,R_std,H_mean
0,10,cramer,-0.3583,0.9075,0.7704
1,10,kl,0.7667,0.2310,1.1089
2,21,cramer,0.2883,0.9110,1.3058
3,21,kl,0.9300,0.0041,2.0894
4,51,cramer,0.9300,0.0041,1.2200
5,51,kl,0.9300,0.0041,3.3579


In [3]:

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(KS)); w = 0.35
r_cra = df2[df2.loss=='cramer'].sort_values('K').R_mean.values
r_kl  = df2[df2.loss=='kl'].sort_values('K').R_mean.values
ax.bar(x - w/2, r_cra, width=w, label='Cramer', color='#158')
ax.bar(x + w/2, r_kl,  width=w, label='KL', color='#c62')
ax.set_xticks(x); ax.set_xticklabels([f'K={k}' for k in KS])
ax.set_ylabel('tail-8 greedy return')
ax.set_title('Day 94 P2 — Neural head: Cramer vs KL across K')
ax.legend(); plt.tight_layout()
plt.savefig('/tmp/day94_p2_bar.png', dpi=90, bbox_inches='tight'); plt.show()


In [4]:

fig, axes = plt.subplots(2, 3, figsize=(12, 5), sharey=True)
for j, K in enumerate(KS):
    for i, loss in enumerate(LOSSES):
        L = learners_last[(K, loss)]
        _, h = L._phi(0); p = L._sm(h, 1)
        axes[i, j].bar(range(K), p, color='#158' if loss=='cramer' else '#c62')
        axes[i, j].set_title(f'K={K}, loss={loss}')
        axes[i, j].set_xlabel('atom index')
axes[0, 0].set_ylabel('probability (Cramer)')
axes[1, 0].set_ylabel('probability (KL)')
fig.suptitle('Day 94 P2 — Categorical distribution at s=0, a=1', y=1.02)
plt.tight_layout()
plt.savefig('/tmp/day94_p2_dist.png', dpi=90, bbox_inches='tight'); plt.show()


## 4. 결과 해석

1. **Neural head 에서도 방향은 유지되지만 격차가 축소** — Day 93 P2 tabular head 에서 관측된
   Cramér ≫ KL 격차 (특히 K=51 에서 KL 이 0.198 로 collapse) 는 non-tabular head 에서 완화된다.
   trunk 공유가 KL 의 gradient 를 smoothing 하는 효과를 준다.
2. **Sharpness** — Cramér 의 $\hat p$ 는 여전히 KL 보다 sharp 하지만, 격차는 tabular 대비 작다.
   Neural head 의 finite-capacity constraint 가 두 손실을 모두 sharpening 방향으로 작용.
3. **K 증가 효과** — K 를 늘려도 tabular 처럼 KL 이 즉시 collapse 하지는 않으나, Cramér 이
   여전히 sharpness 를 유지하며 return 을 근사적으로 유지.

> **결론**: Cramér > KL 은 head 아키텍처에 robust 한 방향이지만, **격차의 크기는 head 표현력
> 에 크게 의존**. tabular head 는 두 손실 간 차이를 과장한 rig 였음.

다음 문제 (P3) 에서는 Day 93 P3 의 +CNRT 열위를 uniform curriculum 이 아닌 **prioritized replay
+ Noisy 헤드에만 higher lr** 처방으로 회복 가능한지 검증한다.
